# 03 — Error Analysis
Confusion matrix + sample misclassifications. If a trained CNN model exists, use it;
otherwise fall back to a quick KNN to demonstrate the workflow.

In [ ]:
%pip -q install scikit-learn seaborn tensorflow==2.* --disable-pip-version-check

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from src.utils import seed_everything, load_train_test, as_images, confusion_matrix_plot, show_misclassified

seed_everything(42)
X, y, _ = load_train_test("data")

In [ ]:
# Try to load a trained CNN model first
use_cnn = False
model_path = "models/cnn_v1.keras"
if os.path.exists(model_path):
    try:
        import tensorflow as tf
        cnn_model = tf.keras.models.load_model(model_path)
        use_cnn = True
        print("Loaded CNN model:", model_path)
    except Exception as e:
        print("Could not load CNN model:", e)

In [ ]:
# Create a hold-out validation split for error analysis
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)

In [ ]:
if use_cnn:
    ypred = np.argmax(cnn_model.predict(as_images(Xva), verbose=0), axis=1)
else:
    knn = KNeighborsClassifier(n_neighbors=3, weights="distance", n_jobs=-1)
    knn.fit(Xtr, ytr)
    ypred = knn.predict(Xva)
acc = (ypred == yva).mean()
print("Validation accuracy:", acc)
print(classification_report(yva, ypred, digits=4))
confusion_matrix_plot(yva, ypred, normalize=True, title="Confusion Matrix (normalized)")
show_misclassified(Xva, yva, ypred, n=16, title="Common mistakes")